In [0]:
import logging
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)


In [0]:

# Config
CATALOG = "oil_gas_demo"
SRC_FILE = f"/Volumes/{CATALOG}/raw/raw_volume/supply_chain_dataset.csv"
BRONZE = f"{CATALOG}.bronze.supply_chain_transactions"


class BronzeLoader:
    """Loads raw CSV data into the bronze layer with audit columns."""

    def __init__(self, spark, file_path: str, table_name: str):
        self.spark = spark
        self.file_path = file_path
        self.table_name = table_name
        self.df = None

    def read_csv(self):
        """Read CSV file with all columns as strings (Spark default)."""
        logger.info(f"Reading CSV from: {self.file_path}")
        self.df = (
            self.spark.read
            .option("header", True)
            .option("multiLine", True)
            .csv(self.file_path)
        )
        logger.info(f"Loaded {self.df.count()} rows from CSV")

    def add_txn_date(self):
        """Parse transaction_date into txn_date (M/d/yyyy with ISO fallback)."""
        logger.info("Parsing transaction_date -> txn_date")
        
        self.df = self.df.withColumns({
            "txn_date": F.coalesce(
                F.expr("try_to_date(transaction_date, 'M/d/yyyy')"),
                F.expr("try_to_date(transaction_date, 'yyyy-MM-dd')"),
            ),
        })

    def add_audit_columns(self):
        """Add ingestion timestamp and source file path."""
        logger.info("Adding audit columns: _ingest_ts, _source_file")
        self.df = self.df.withColumns({
            "_ingest_ts": F.current_timestamp(),
            "_source_file": F.col("_metadata.file_path"),
        })

    def write_to_bronze(self):
        """Write the DataFrame to the bronze Delta table."""
        logger.info(f"Writing to bronze table: {self.table_name}")
        (
            self.df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(self.table_name)
        )
        logger.info(f"Bronze table written successfully: {self.table_name}")



    def run(self):
        """Execute the full bronze pipeline."""
        self.read_csv()
        self.add_txn_date()
        self.add_audit_columns()
        self.write_to_bronze()


bronze = BronzeLoader(
    spark=spark,
    file_path=SRC_FILE,
    table_name=BRONZE,
)
bronze.run()
